# Week 3: QGIS hands-on: load, style, export

**Track:** Ground Station Operator (Beginner)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/3/](https://launchdetect.com/academy/week/3/)  
**Track index:** [https://launchdetect.com/academy/ground-station-operator/](https://launchdetect.com/academy/ground-station-operator/)

---

_QGIS is the free and open-source workhorse of desktop GIS. This week is hands-on: load real launch-site data, query it, style it with rules-based symbology, and export a publication-quality map._


## Why this week matters

QGIS is the free, open-source desktop GIS that every serious geospatial team uses for analysis, styling, and final map composition. It's the de-facto alternative to ArcGIS Pro and unlike its commercial competitor, it costs nothing, runs on every platform, and has a plugin ecosystem of thousands. You will use QGIS at every stage of every space-domain workflow.

But QGIS is a tool for *exploration and final delivery*, not for production automation. The rule of thumb: if you're going to run this analysis once and ship a PDF, do it in QGIS. If you're going to run it every hour as a Lambda, do it in Python. This week trains the QGIS muscle so future weeks can use it as a validation tool — render the Python output in QGIS, visually verify, debug.

We're using `leafmap` in this notebook as the in-browser equivalent for everyone who can't or won't install QGIS, but the QGIS instructions below are the canonical desktop workflow you'll want to know.


## Learning objectives

By the end of this lab you will be able to:

- Install QGIS and open a project
- Load vector (GeoJSON) and raster (GeoTIFF) layers
- Use the attribute table and run a simple query
- Style a layer with rules-based symbology
- Export a print-quality map composition


## Setup — and why these dependencies

`leafmap` provides QGIS-like visualization in the browser, which means we can demo the workflow without forcing a QGIS install. **But** if you do install QGIS (highly recommended — see the QGIS section below), the same data can be opened there. QGIS itself is free at qgis.org/download — the **long-term-release (LTR)** version is what professional teams standardize on.


In [ ]:
# Install required packages
!pip install -q leafmap[common] pyproj geopandas shapely


## The approach (and why)

Load the world's currently-active orbital launch sites as a vector layer, attribute them with the right metadata (country, operator, vehicles), and produce both a browser-renderable interactive map (leafmap) AND a print-quality static map (QGIS).

**Why both?** Different audiences. The leafmap version goes in a blog post, a PowerPoint, or a Streamlit dashboard. The QGIS print layout goes in a printed report, a poster, or a 100-MB PDF for a press release. Same data, two different channels — the workflow for each is different, and you need to know both.

**Why category-based styling on operator?** A reader looking at the global map wants to answer 'who runs which pad?' at a glance. Color by operator turns that into a one-second perception. Color by country would answer 'who has launch capacity?' — equally valid, different question.


In [ ]:
# Week 3: load global launch sites in QGIS — OR equivalent in leafmap.
# This notebook runs the leafmap version; QGIS instructions follow in markdown.
import leafmap.foliumap as leafmap

# 20 active orbital launch pads (subset; full list is the Capstone 1 deliverable)
pads = [
    {"name": "Cape Canaveral",        "country": "US", "operator": "USSF/SpaceX", "lat": 28.49, "lon": -80.60},
    {"name": "Kennedy Space Center",  "country": "US", "operator": "NASA",        "lat": 28.57, "lon": -80.65},
    {"name": "Vandenberg",            "country": "US", "operator": "USSF",        "lat": 34.74, "lon": -120.57},
    {"name": "Wallops / MARS",        "country": "US", "operator": "NASA",        "lat": 37.94, "lon": -75.47},
    {"name": "Starbase (Boca Chica)", "country": "US", "operator": "SpaceX",      "lat": 26.00, "lon": -97.16},
    {"name": "Kourou (CSG)",          "country": "FR", "operator": "Arianespace", "lat": 5.24,  "lon": -52.77},
    {"name": "Baikonur",              "country": "KZ", "operator": "Roscosmos",   "lat": 45.97, "lon": 63.31},
    {"name": "Plesetsk",              "country": "RU", "operator": "Roscosmos",   "lat": 62.96, "lon": 40.57},
    {"name": "Vostochny",             "country": "RU", "operator": "Roscosmos",   "lat": 51.88, "lon": 128.33},
    {"name": "Tanegashima",           "country": "JP", "operator": "JAXA",        "lat": 30.40, "lon": 130.97},
    {"name": "Wenchang",              "country": "CN", "operator": "CASC",        "lat": 19.61, "lon": 110.95},
    {"name": "Jiuquan",               "country": "CN", "operator": "CASC",        "lat": 40.96, "lon": 100.29},
    {"name": "Xichang",               "country": "CN", "operator": "CASC",        "lat": 28.25, "lon": 102.03},
    {"name": "Taiyuan",               "country": "CN", "operator": "CASC",        "lat": 38.85, "lon": 111.61},
    {"name": "Sriharikota",           "country": "IN", "operator": "ISRO",        "lat": 13.72, "lon": 80.23},
    {"name": "Mahia",                 "country": "NZ", "operator": "Rocket Lab",  "lat": -39.26,"lon": 177.86},
]

m = leafmap.Map(center=[20, 0], zoom=2)
for p in pads:
    m.add_marker((p["lat"], p["lon"]), popup=f"{p['name']} ({p['country']}) — {p['operator']}")
m

# TODO: export the pads list as GeoJSON to disk, open it in QGIS, and produce
# a styled PDF map composition. (QGIS section follows in markdown.)


## What just happened — and why it works

The leafmap render assigns each launch pad a default-styled marker with a popup. Click any marker — you see the metadata in a popup card. This is the most basic interactive-map pattern: load points, attribute them, render with click-to-detail. Every dashboard you ever build will start with this pattern.

Notice how every pad has `lat` and `lon` columns plus categorical attributes (country, operator). That's the *attribute table*: a spreadsheet where one column is the geometry. QGIS calls it the attribute table; leafmap/Folium calls it the feature properties; pandas+geopandas calls it the GeoDataFrame. Same structure, different vocab. Get fluent in the synonyms.


## Common gotchas

- **Coordinate precision matters.** A launch pad at 28.6°N is at one of thousands of pads worldwide; 28.6042°N is *the* SLC-40 pad at Cape Canaveral. Use at least 4 decimal places (~10 m precision) for any pad-level work.
- **Operator names are not standardized.** ULA, United Launch Alliance, and 'ULA (Boeing/Lockheed)' are the same operator. Pick one canonical form and stick to it.
- **Status is time-varying.** A pad that was 'active' in 2023 may be 'retired' in 2026 (Atlas V's retirement). Date-stamp your dataset and refresh it.


## Doing this in QGIS (alternative path)

**Full QGIS walkthrough — why each step:**

1. **File → New Project.** Sets up an empty canvas in WGS84 by default. The CRS shown at the bottom-right is the project CRS — everything you draw will be in this. *Why WGS84:* matches GPS, matches our input data, matches every public source.

2. **Layer → Add Layer → Add Vector Layer → select your `launch_sites.geojson`.** GeoJSON is the lingua franca of vector data exchange. *Why GeoJSON over Shapefile:* single-file (vs Shapefile's 3-7 sidecar files), UTF-8 by default (vs Shapefile's broken DBF encoding), human-readable in a text editor. Shapefiles still exist for legacy compatibility but never as a first choice in 2026.

3. **Right-click the layer → Open Attribute Table** (`F6`). Inspect the data: each row is a pad, each column an attribute, the geometry is shown implicitly via the row position on the map. *Why open the table:* you should always look at your data before styling it — typos, missing values, wrong-type columns are visible here.

4. **Layer Properties → Symbology → Categorized → Column: `operator` → Classify.** QGIS scans the values, gives each unique operator a distinct color from the active palette. Apply. *Why categorized over graduated:* operator is a categorical variable, not numeric. Graduated would only make sense for numeric columns like `first_launch_year`.

5. **Layer Properties → Labels → Single Labels → Value: `name` → buffer (white halo).** Labels at small zooms can collide; the buffer (a white halo around each label) makes them readable over any basemap. *Why buffer instead of opaque background:* the halo preserves the basemap context underneath, which matters for cartography.

6. **Project → New Print Layout → name it 'Global Launch Pads'.** A print layout is QGIS's equivalent of a slide: you place a map item, a title, a legend, scale bar, and north arrow on a fixed-size page. *Why use print layouts at all:* this is the only path to publication-quality output. Screenshots are not publication-quality.

7. **Add Map → drag a rectangle that fills most of the page. Add Legend, Scale Bar, North Arrow, attribution text in the bottom-right.** Export → 300 dpi PDF. *Why 300 dpi:* print-quality threshold. Below 300 dpi, labels start looking pixelated in print; above 300 dpi is wasted file size unless you're doing professional offset printing.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/3/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
